In [ ]:
import pandas as pd
df = pd.read_csv("demanddata_2025.csv")


In [ ]:
df.head()

,SETTLEMENT_DATE,SETTLEMENT_PERIOD,ND,TSD,ENGLAND_WALES_DEMAND,EMBEDDED_WIND_GENERATION,EMBEDDED_WIND_CAPACITY,EMBEDDED_SOLAR_GENERATION,EMBEDDED_SOLAR_CAPACITY,NON_BM_STOR,...,IFA_FLOW,IFA2_FLOW,BRITNED_FLOW,MOYLE_FLOW,EAST_WEST_FLOW,NEMO_FLOW,NSL_FLOW,ELECLINK_FLOW,VIKING_FLOW,GREENLINK_FLOW
0,2025-01-01,1,21036,26215,19918,5405,6606,0,19270,0,...,-1518,-1026,1,-232,-386,-558,96,0,241,0
1,2025-01-01,2,21222,26063,20167,5358,6606,0,19270,0,...,-1513,-1023,1,-124,-282,-564,51,0,262,0
2,2025-01-01,3,21385,25734,20328,5237,6606,0,19270,0,...,-1428,-922,1,-119,-384,-205,62,0,414,0
3,2025-01-01,4,20473,24988,19460,5115,6606,0,19270,0,...,-1423,-915,0,-119,-334,-193,37,0,423,0
4,2025-01-01,5,19634,24912,18654,5080,6606,0,19270,0,...,-1539,-899,0,-119,-363,-622,-282,0,288,0


In [ ]:
DATA_PATH = "demanddata_2025.csv"

In [ ]:
"""
Probabilistic forecasting of UK electricity demand:
quantile regression (thesis baseline) vs modern machine learning.

Author: Titilayo Adewale
Purpose: short recent research project extending the MTech thesis
         ("Determinants of stock market index in Nigeria: quantile
         regression approach", FUTA, 2018) to machine learning.

Research question
-----------------
Do machine learning quantile models give better calibrated and sharper
prediction intervals for half-hourly UK electricity demand than classical
linear quantile regression?

How to run in Google Colab
--------------------------
1. Download a full year of "Historic Demand Data" (CSV) from NESO
   (data.nationalgrideso.com). Upload it to Colab (left panel > Files).
2. Set demanddata below to the file name.
3. Run the whole file (Runtime > Run all). If there is no file, the script
   generates synthetic demo data so you can test that everything works.
   Do NOT report results from synthetic data.

Models compared (each fitted at quantiles 0.1, 0.5 and 0.9)
-----------------------------------------------------------
1. Linear quantile regression (statsmodels)         <- thesis baseline
2. Gradient boosting with quantile loss (scikit-learn)
3. Random forest quantile estimate (scikit-learn, from per-tree spread)
   NOTE: the spread of tree predictions mostly measures model variance, not
   the full noise in demand, so its intervals are usually too narrow and
   under-cover. This is a known limitation and a good discussion point.
   A proper quantile regression forest (e.g. the quantile-forest package)
   is a natural next step.

Metrics
-------
- Pinball loss at each quantile (lower is better)
- Coverage of the 80% prediction interval (target = 0.80)
- Mean width of the 80% interval (narrower is sharper, if coverage holds)
"""

import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor

warnings.filterwarnings("ignore")

# ----------------------------------------------------------------------------
# 0. Settings
# ----------------------------------------------------------------------------
DATA_PATH = "demanddata_2025.csv"      # change to your NESO file name
DEMAND_COL = None             # None = auto-detect (see load_data)
DATE_COL = None               # None = auto-detect
PERIOD_COL = None             # settlement period column (1-48), None = auto
QUANTILES = [0.1, 0.5, 0.9]
TEST_FRACTION = 0.2           # last 20% of the year held out (time order kept)
SEED = 42
OUTDIR = "results"
os.makedirs(OUTDIR, exist_ok=True)
rng = np.random.default_rng(SEED)


# ----------------------------------------------------------------------------
# 1. Data loading
# ----------------------------------------------------------------------------
def make_synthetic(n_days=365):
    """Synthetic half-hourly demand for testing the pipeline only."""
    idx = pd.date_range("2025-01-01", periods=n_days * 48, freq="30min")
    hour = idx.hour + idx.minute / 60
    daily = 6000 * np.sin((hour - 6) / 24 * 2 * np.pi) ** 2 + 18000
    season = 4000 * np.cos((idx.dayofyear - 15) / 365 * 2 * np.pi)
    weekend = np.where(idx.dayofweek >= 5, -2500, 0)
    noise_scale = 600 + 300 * (season > 0)
    noise = rng.normal(0, 1, len(idx)) * noise_scale
    demand = daily + season + weekend + noise
    return pd.DataFrame({"timestamp": idx, "demand": demand})


def load_data(path):
    if not os.path.exists(path):
        print(f"[!] {path} not found. Using SYNTHETIC data for a pipeline test.")
        return make_synthetic(), True

    raw = pd.read_csv(path)
    cols = {c.lower().strip(): c for c in raw.columns}

    date_col = DATE_COL or next(
        (cols[c] for c in cols if c in ("settlement_date", "date", "datetime", "timestamp")),
        None,
    )
    period_col = PERIOD_COL or next(
        (cols[c] for c in cols if "period" in c), None
    )
    demand_col = DEMAND_COL or next(
        (cols[c] for c in cols if c in ("nd", "national_demand", "england_wales_demand", "tsd", "demand")),
        None,
    )
    if demand_col is None:
        raise ValueError(f"Could not find a demand column. Columns: {list(raw.columns)}. "
                         "Set DEMAND_COL at the top of the script.")
    if date_col is None:
        raise ValueError(f"Could not find a date column. Columns: {list(raw.columns)}. "
                         "Set DATE_COL at the top of the script.")

    df = pd.DataFrame({"demand": pd.to_numeric(raw[demand_col], errors="coerce")})
    # ISO dates (2025-03-04) parse correctly by default. Only fall back to
    # day-first parsing if most dates fail (e.g. 04/03/2025 style files).
    date = pd.to_datetime(raw[date_col], errors="coerce")
    if date.isna().mean() > 0.5:
        date = pd.to_datetime(raw[date_col], errors="coerce", dayfirst=True)
    if period_col is not None:
        period = pd.to_numeric(raw[period_col], errors="coerce")
        df["timestamp"] = date + pd.to_timedelta((period - 1) * 30, unit="m")
    else:
        df["timestamp"] = date
    df = df.dropna().sort_values("timestamp").drop_duplicates("timestamp")
    print(f"Loaded {len(df):,} rows from {path} "
          f"({df.timestamp.min():%Y-%m-%d} to {df.timestamp.max():%Y-%m-%d}); "
          f"demand column = '{demand_col}'")
    return df[["timestamp", "demand"]].reset_index(drop=True), False


# ----------------------------------------------------------------------------
# 2. Features (known in advance: calendar + lagged demand)
# ----------------------------------------------------------------------------
def build_features(df):
    d = df.copy().set_index("timestamp")
    d["hour"] = d.index.hour + d.index.minute / 60
    d["dow"] = d.index.dayofweek
    d["is_weekend"] = (d["dow"] >= 5).astype(int)
    d["doy"] = d.index.dayofyear
    # cyclical encodings so 23:30 and 00:00 are close together
    d["hour_sin"] = np.sin(2 * np.pi * d["hour"] / 24)
    d["hour_cos"] = np.cos(2 * np.pi * d["hour"] / 24)
    d["doy_sin"] = np.sin(2 * np.pi * d["doy"] / 365.25)
    d["doy_cos"] = np.cos(2 * np.pi * d["doy"] / 365.25)
    # lags: same time yesterday and last week (day-ahead forecast is realistic)
    d["lag_48"] = d["demand"].shift(48)
    d["lag_336"] = d["demand"].shift(336)
    d = d.dropna()
    features = ["hour_sin", "hour_cos", "doy_sin", "doy_cos",
                "is_weekend", "dow", "lag_48", "lag_336"]
    return d, features


# ----------------------------------------------------------------------------
# 3. Metrics
# ----------------------------------------------------------------------------
def pinball(y, q_pred, tau):
    diff = y - q_pred
    return np.mean(np.maximum(tau * diff, (tau - 1) * diff))


def evaluate(name, y, preds):
    """preds: dict {tau: array}"""
    row = {"model": name}
    for tau in QUANTILES:
        row[f"pinball_{tau}"] = pinball(y, preds[tau], tau)
    lo, hi = preds[QUANTILES[0]], preds[QUANTILES[-1]]
    row["coverage_80"] = float(np.mean((y >= lo) & (y <= hi)))
    row["mean_width_80"] = float(np.mean(hi - lo))
    row["avg_pinball"] = np.mean([row[f"pinball_{t}"] for t in QUANTILES])
    return row


# ----------------------------------------------------------------------------
# 4. Models
# ----------------------------------------------------------------------------
def fit_linear_quantile(Xtr, ytr, Xte):
    Xtr_c, Xte_c = sm.add_constant(Xtr), sm.add_constant(Xte, has_constant="add")
    out = {}
    for tau in QUANTILES:
        res = sm.QuantReg(ytr, Xtr_c).fit(q=tau, max_iter=5000)
        out[tau] = res.predict(Xte_c)
    return out


def fit_gbm_quantile(Xtr, ytr, Xte):
    out = {}
    for tau in QUANTILES:
        m = GradientBoostingRegressor(
            loss="quantile", alpha=tau, n_estimators=300, max_depth=4,
            learning_rate=0.05, subsample=0.8, random_state=SEED,
        )
        m.fit(Xtr, ytr)
        out[tau] = m.predict(Xte)
    return out


def fit_rf_quantile(Xtr, ytr, Xte):
    """Quantiles from the spread of the individual tree predictions.
    Simple approximation of a quantile regression forest."""
    rf = RandomForestRegressor(
        n_estimators=200, min_samples_leaf=10, n_jobs=-1, random_state=SEED
    )
    rf.fit(Xtr, ytr)
    tree_preds = np.column_stack([t.predict(Xte.values) for t in rf.estimators_])
    return {tau: np.quantile(tree_preds, tau, axis=1) for tau in QUANTILES}


# ----------------------------------------------------------------------------
# 5. Run
# ----------------------------------------------------------------------------
def main():
    df, synthetic = load_data(DATA_PATH)
    data, features = build_features(df)

    split = int(len(data) * (1 - TEST_FRACTION))
    train, test = data.iloc[:split], data.iloc[split:]
    Xtr, ytr = train[features], train["demand"].values
    Xte, yte = test[features], test["demand"].values
    print(f"Train: {len(train):,} rows | Test: {len(test):,} rows "
          f"(time-ordered split, no shuffling)")

    results, all_preds = [], {}
    for name, fn in [
        ("Linear quantile regression (thesis baseline)", fit_linear_quantile),
        ("Gradient boosting (quantile loss)", fit_gbm_quantile),
        ("Random forest (tree-spread quantiles)", fit_rf_quantile),
    ]:
        print(f"Fitting: {name} ...")
        preds = fn(Xtr, ytr, Xte)
        all_preds[name] = preds
        results.append(evaluate(name, yte, preds))

    res = pd.DataFrame(results).round(3)
    res.to_csv(f"{OUTDIR}/metrics.csv", index=False)
    print("\n=== Results (lower pinball loss is better; coverage target 0.80) ===")
    print(res.to_string(index=False))
    if synthetic:
        print("\n[!] These results come from SYNTHETIC data. Do not report them.")

    # --- Chart 1: one week of test data with the 80% interval ---------------
    week = slice(0, 48 * 7)
    fig, axes = plt.subplots(len(all_preds), 1, figsize=(11, 9), sharex=True)
    for ax, (name, preds) in zip(np.atleast_1d(axes), all_preds.items()):
        t = test.index[week]
        ax.fill_between(t, preds[QUANTILES[0]][week], preds[QUANTILES[-1]][week],
                        alpha=0.3, label="80% interval")
        ax.plot(t, preds[0.5][week], lw=1, label="median forecast")
        ax.plot(t, yte[week], lw=0.8, color="black", label="actual")
        ax.set_title(name, fontsize=10)
        ax.set_ylabel("Demand (MW)")
    np.atleast_1d(axes)[0].legend(loc="upper right", fontsize=8)
    plt.tight_layout()
    plt.savefig(f"{OUTDIR}/interval_forecasts.png", dpi=150)
    plt.close()

    # --- Chart 2: calibration (coverage at each quantile) ---------------------
    fig, ax = plt.subplots(figsize=(6, 5))
    for name, preds in all_preds.items():
        obs = [np.mean(yte <= preds[t]) for t in QUANTILES]
        ax.plot(QUANTILES, obs, marker="o", label=name)
    ax.plot([0, 1], [0, 1], "k--", lw=1, label="perfect calibration")
    ax.set_xlabel("Nominal quantile")
    ax.set_ylabel("Observed share of actuals below forecast")
    ax.set_title("Calibration on held-out data")
    ax.legend(fontsize=7)
    plt.tight_layout()
    plt.savefig(f"{OUTDIR}/calibration.png", dpi=150)
    plt.close()

    # --- Chart 3: pinball loss comparison -------------------------------------
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.barh(res["model"], res["avg_pinball"])
    ax.set_xlabel("Average pinball loss (lower is better)")
    ax.invert_yaxis()
    plt.tight_layout()
    plt.savefig(f"{OUTDIR}/pinball_comparison.png", dpi=150)
    plt.close()

    print(f"\nSaved metrics.csv and 3 charts to ./{OUTDIR}/")


if __name__ == "__main__":
    main()

Loaded 17,518 rows from demanddata_2025.csv (2025-01-01 to 2025-12-31); demand column = 'ND'
Train: 13,745 rows | Test: 3,437 rows (time-ordered split, no shuffling)
Fitting: Linear quantile regression (thesis baseline) ...
Fitting: Gradient boosting (quantile loss) ...
Fitting: Random forest (tree-spread quantiles) ...

=== Results (lower pinball loss is better; coverage target 0.80) ===
                                       model  pinball_0.1  pinball_0.5  pinball_0.9  coverage_80  mean_width_80  avg_pinball
Linear quantile regression (thesis baseline)      381.195      813.221      365.215        0.772       5157.265      519.877
           Gradient boosting (quantile loss)      621.426      896.366      347.890        0.548       3150.327      621.894
       Random forest (tree-spread quantiles)      440.572      805.670      397.927        0.525       3141.626      548.056

Saved metrics.csv and 3 charts to ./results/
